In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

df = pd.read_csv("synthetic_core_pc_dataset.csv")

NUM_POINTS = 10
COMMON_POINTS = 50
common_sw = np.linspace(0.15, 0.95, COMMON_POINTS)

long_dataset = []

for index, row in df.iterrows():
    # Retrieve the explicitly generated scaling factor
    scaling = row["Target_Scaling_Factor"]
    
    Sw = np.array([row[f"Sw_{i}"] for i in range(1, NUM_POINTS + 1)], dtype=float)
    Pc = np.array([row[f"Pc_{i}"] for i in range(1, NUM_POINTS + 1)], dtype=float)

    sort_idx = np.argsort(Sw)
    Sw, Pc = Sw[sort_idx], Pc[sort_idx]
    Sw_unique, unique_idx = np.unique(Sw, return_index=True)
    Pc_unique = Pc[unique_idx]

    interpolation = interp1d(Sw_unique, Pc_unique, kind="linear", bounds_error=False, fill_value=(Pc_unique[0], Pc_unique[-1]))
    Pc_interp = np.nan_to_num(interpolation(common_sw), nan=Pc_unique[-1], posinf=Pc_unique[-1], neginf=Pc_unique[0])
    
    Pc_upscaled = Pc_interp * scaling

    if np.any(np.isnan(Pc_interp)) or np.any(np.isinf(Pc_interp)):
        continue

    for j in range(COMMON_POINTS):
        long_dataset.append({
            "Curve_ID": row["Curve_ID"],
            "Sw": common_sw[j],
            "Core_Pc": Pc_interp[j],
            "Upscaled_Pc": Pc_upscaled[j],
            "Core_Porosity": row["Core_Porosity"],
            "Core_Permeability_mD": row["Core_Permeability_mD"],
            "Reservoir_Porosity": row["Reservoir_Porosity"],
            "Reservoir_Permeability_mD": row["Reservoir_Permeability_mD"],
            "Lithology": row["Lithology"],
            "Target_Scaling_Factor": scaling
        })

long_df = pd.DataFrame(long_dataset)
long_df.to_csv("core_upscaled_dataset_long.csv", index=False)
print(f"Long dataset saved with {len(long_df)} rows.")

Long dataset saved with 250000 rows.
